# Functions for cross tabulated format (CTF) 2

Created: 2026-11-10

The previous notebook in this series [Functions for cross tabulated format (CTF) 1](xlsx-ctf-functions-01.ipynb) served for developing a series of functions for processing Excel files from e-Stat that are in conventional cross tablulated format (CTF). This notebook is for developing the top level processing function for getting a data frame starting from a file path. It will also serve for hunting bugs using files from several releases in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

Functionality for column-oriented Excel will be developed later.

## Install the necessary packages and development version of estatjp

In [ ]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install openpyxl
%pip install treelib

## Data source urls

Workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

In [6]:
dataset = "201212"
url12120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315227&fileKind=0"
url12120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315228&fileKind=0"
url12120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315229&fileKind=0"
url12120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315230&fileKind=0"
url121203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315231&fileKind=0"
url121204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315232&fileKind=0"
url12120500 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315233&fileKind=0"
url12120501 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315234&fileKind=0"
url12120502 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315235&fileKind=0"
url12120503 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315236&fileKind=0"
url12120504 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315237&fileKind=0"
url12120505 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315238&fileKind=0"
url12120600 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315239&fileKind=0"
url12120601 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315240&fileKind=0"
url12120602 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315241&fileKind=0"
url12120603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315242&fileKind=0"
url12120604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315303&fileKind=0"
url12120605 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315304&fileKind=0"
url121207 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000022401560&fileKind=0"


In [3]:
dataset = "201306"
url13060101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609633&fileKind=0"
url13060102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609746&fileKind=0"
url13060201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609634&fileKind=0"
url13060202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609752&fileKind=0"
url130603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609753&fileKind=0"
url130604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609653&fileKind=0"
url13060500 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609747&fileKind=0"
url13060501 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609654&fileKind=0"
url13060502 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609635&fileKind=0"
url13060503 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609772&fileKind=0"
url13060504 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609754&fileKind=0"
url13060505 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609748&fileKind=0"
url13060600 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609656&fileKind=0"
url13060601 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609773&fileKind=0"
url13060602 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609755&fileKind=0"
url13060603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609655&fileKind=0"
url13060604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609749&fileKind=0"
url13060605 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609657&fileKind=0"
url130607 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000023609774&fileKind=0"


## Import the module and download a file

In [7]:
from estatjp.xlsx import xlsx
from pathlib import Path

theurl = url121207
ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw/isa") / dataset
if Path.exists(dataraw) == False:
    Path.mkdir(dataraw)
dataprocessed = ROOT_DIR / Path(".data/processed/isa") / dataset
if Path.exists(dataprocessed) == False:
    Path.mkdir(dataprocessed)
newdata = xlsx.download_to_folder(theurl, dataraw, overwrite=True, verbose=True)
print('source: ' + str(newdata))
if newdata != None:
    processeddata = dataprocessed / newdata.name
    processeddata = processeddata.with_suffix(".csv")
    print('target: ' + str(processeddata))

{'Content-Type': 'application/octet-stream', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'X-Cache': 'MISS', 'Content-Encoding': 'gzip', 'Accept-Ranges': 'bytes', 'Content-Disposition': "attachment; filename*=UTF-8''12-12-07.xlsx", 'X-Content-Type-Options': 'nosniff', 'Date': 'Sat, 14 Nov 2026 08:09:21 GMT', 'X-Zen-Fury': '4efd4ed0b097e7e2bb58db6e5f136f1a915ac71c', 'Server': 'ZENEDGE', 'X-XSS-Protection': '1; mode=block', 'X-Frame-Options': 'SAMEORIGIN', 'Strict-Transport-Security': 'max-age=0', 'X-Cache-Status': 'NOTCACHED', 'Cache-Control': 'private', 'X-Cdn': 'Served-By-Zenedge'}
source: c:\Users\Alan\Documents\GitHub\estatjp\.data\raw\isa\201212\12-12-07.xlsx
target: c:\Users\Alan\Documents\GitHub\estatjp\.data\processed\isa\201212\12-12-07.csv


## Main function call for this notebook

In [8]:
import csv

df = xlsx.dataframe_from_xlsx(xlsxpath=newdata, verbose=False)

df.to_csv(processeddata, index=False, quoting=csv.QUOTE_ALL)
    #df
print(df.to_string())

      table_name   time_code stats_date title_population sub_population  cat01 cat01lvl0label cat01lvl0 cat01lvl1label cat01lvl1 cat01lvl2label cat01lvl2 units cat02_label cat02lvl0label cat02lvl0 cat02lvl1label cat02lvl1     value
0       12-12-07  2012001212   2012年12月            在留外国人           <NA>  国籍・地域             地域       北海道             国籍       札幌市            階級2      <NV>     人        市区町村           市区町村      <NA>            階級1        総数    9562.0
1       12-12-07  2012001212   2012年12月            在留外国人           <NA>  国籍・地域             地域       北海道             国籍       札幌市            階級2      <NV>     人        市区町村           市区町村      <NA>            階級1        中国    3741.0
2       12-12-07  2012001212   2012年12月            在留外国人           <NA>  国籍・地域             地域       北海道             国籍       札幌市            階級2      <NV>     人        市区町村           市区町村      <NA>            階級1        台湾     126.0
3       12-12-07  2012001212   2012年12月            在留外国人           <NA> 

In [ ]:
import re

CAT1_LABEL_REGEX = re.compile(r"((国籍)・(地域))")
CAT2_LABEL_REGEX = re.compile(r"((\w+)（(\w+)）)|((年齢)・(男女))")
m =CAT2_LABEL_REGEX.match("年齢・男女")
m =CAT2_LABEL_REGEX.match("在留資格（在留目的）")
m =CAT1_LABEL_REGEX.match("国籍・地域")

print(m.groups())